In [ ]:


import os
import glob
import json
from typing import Dict

import uvicorn
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

from langchain.llms import OpenAI
from langchain.vectorstores import FAISS
from langchain.document_loaders import TextLoader
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory


OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", None)
if not OPENAI_API_KEY:
    raise RuntimeError("Please set the OPENAI_API_KEY environment variable")

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

DOCS_PATH = "./docs/**/*.txt"   # folder containing FAQs, manuals, help docs
VECTOR_STORE_PATH = "./faiss_index"


session_chains: Dict[str, ConversationalRetrievalChain] = {}

class ChatRequest(BaseModel):
    session_id: str          
    role: str                
    message: str

class ChatResponse(BaseModel):
    answer: str
    escalate: bool

def build_vector_store() -> FAISS:
    # Load and chunk all docs
    loader = TextLoader
    all_texts = []
    for fname in glob.glob(DOCS_PATH, recursive=True):
        docs = loader(fname).load()
        all_texts.extend(docs)

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=100
    )
    docs = splitter.split_documents(all_texts)

    embeddings = OpenAIEmbeddings()
    vectordb = FAISS.from_documents(docs, embeddings)
    vectordb.save_local(VECTOR_STORE_PATH)
    return vectordb

def load_vector_store() -> FAISS:
    if os.path.isdir(VECTOR_STORE_PATH):
        return FAISS.load_local(VECTOR_STORE_PATH, OpenAIEmbeddings())
    else:
        return build_vector_store()

vectordb = load_vector_store()
retriever = vectordb.as_retriever(search_kwargs={"k": 4})
def get_chain_for_session(session_id: str, role: str) -> ConversationalRetrievalChain:
    """
    Create or retrieve a chain for a given session_id.
    """
    if session_id in session_chains:
        return session_chains[session_id]

    system_prompt = (
        f"You are Kelunga's virtual assistant. You help a {role} with "
        "platform flow, bookings, profiles, and services. Always be polite."
    )

    llm = OpenAI(temperature=0.2, model_name="gpt-4")

    memory = ConversationBufferMemory(
        memory_key="chat_history",
        return_messages=True
    )

    chain = ConversationalRetrievalChain.from_llm(
        llm=llm,
        retriever=retriever,
        memory=memory,
        system_prompt=system_prompt,
        verbose=False
    )
    session_chains[session_id] = chain
    return chain

ESCALATION_KEYWORDS = [
    "escalate",
    "human",
    "support agent",
    "cannot",
    "not sure"
]

def check_escalation(answer: str) -> bool:
    """Detect if we should hand off to human support."""
    lower = answer.lower()
    for kw in ESCALATION_KEYWORDS:
        if kw in lower:
            return True
    return False

def log_conversation(session_id: str, role: str, user_msg: str, bot_reply: str):
    entry = {
        "session_id": session_id,
        "role": role,
        "user": user_msg,
        "bot": bot_reply
    }
    with open("conversation_logs.jsonl", "a", encoding="utf-8") as f:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")


app = FastAPI(title="Kelunga LLM Chatbot")

@app.post("/chat", response_model=ChatResponse)
async def chat(req: ChatRequest):
    if req.role.lower() not in {"seeker", "expert"}:
        raise HTTPException(status_code=400, detail="Role must be 'seeker' or 'expert'")

    chain = get_chain_for_session(req.session_id, req.role.lower())
    result = chain({"question": req.message})

    answer = result.get("answer", "").strip()
    escalate = check_escalation(answer)

    log_conversation(req.session_id, req.role, req.message, answer)
    return ChatResponse(answer=answer, escalate=escalate)

@app.get("/health")
async def health():
    return {"status": "ok"}

if __name__ == "__main__":
    uvicorn.run("main:app", host="0.0.0.0", port=8000, reload=True)


In [1]:
import os
from fastapi import FastAPI, HTTPException, Query
from pydantic import BaseModel
from enum import Enum
from typing import List, Optional

from langchain.chat_models import ChatOpenAI
from langchain.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain.schema import ChatMessage
from langchain.memory import ConversationBufferMemory
from langchain.document_loaders import DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain.embeddings import OpenAIEmbeddings
from langchain.chains import RetrievalQA

env_path = ".env"
if os.path.exists(env_path):
    from dotenv import load_dotenv
    load_dotenv(env_path)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise RuntimeError("Missing OpenAI API key. Set OPENAI_API_KEY in environment.")

app = FastAPI(title="Kelunga LLM-Integrated Smart Chatbot")


class Role(str, Enum):
    seeker = "seeker"
    expert = "expert"


class ChatRequest(BaseModel):
    role: Role
    message: str
    session_id: Optional[str] = None

class ChatResponse(BaseModel):
    reply: str
    escalation: bool = False


llm = ChatOpenAI(
    openai_api_key=OPENAI_API_KEY,
    model_name="gpt-3.5-turbo",
    temperature=0.2,
)


loader = DirectoryLoader("faqs", glob="**/*.txt")# Load nd index of FAQs and manuals
docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
split_docs = splitter.split_documents(docs)
embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)
vector_store = Chroma.from_documents(split_docs, embeddings, collection_name="kelunga_faqs")
retriever = vector_store.as_retriever(search_kwargs={"k": 3})
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=False,
)

memory_store = {}

@app.post("/chat", response_model=ChatResponse)
def chat(request: ChatRequest):
    session_id = request.session_id or "default"
    # Initialize memory if first message
    if session_id not in memory_store:
        memory_store[session_id] = ConversationBufferMemory(
            memory_key="chat_history", return_messages=True
        )

    memory: ConversationBufferMemory = memory_store[session_id]

    
    system_msg = SystemMessagePromptTemplate.from_template(
        "You are a helpful assistant for Kelunga platform. You are chatting with a {role}."
    )
    human_msg = HumanMessagePromptTemplate.from_template(
        "{input}")
    prompt = ChatPromptTemplate.from_messages([system_msg, human_msg])

    
    escalation_keywords = ["human", "agent", "support", "escalate", "representative"]
    if any(kw in request.message.lower() for kw in escalation_keywords):
        return {"reply": "I am escalating your request to our human support team. They'll reach out shortly.", "escalation": True}
    
    retriever_answer = qa_chain.run(request.message)
    if retriever_answer and len(retriever_answer.strip()) > 0:
        response_text = retriever_answer
    else:
        chat_messages: List[ChatMessage] = []
        chat_messages.extend(memory.load_memory_messages())

        
        chat_messages.append(ChatMessage(role="user", content=request.message))
        
        result = llm.generate_messages([prompt.format_prompt(
            role=request.role.value,
            input=request.message
        ).to_messages()])
        response_text = result.generations[0][0].text

    
    memory.chat_memory.add_user_message(request.message)
    memory.chat_memory.add_ai_message(response_text)

    return {"reply": response_text, "escalation": False}

@app.post("/reset_session")
def reset_session(session_id: Optional[str] = Query(None)):
    """Reset the conversation memory for a given session."""
    sid = session_id or "default"
    if sid in memory_store:
        del memory_store[sid]
    return {"status": "reset", "session_id": sid}


  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
   ---------------------------------------- 0.0/3.1 MB ? eta -:--:--
   ---------------------------------------- 3.1/3.1 MB 20.6 MB/s eta 0:00:00
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11704 sha256=d876e7d133bd116a365bf02f2ce33276a7e55199371226158588747f5e48765e
  Stored in directory: c:\users\dell\appdata\local\pip\cache\wheels\79\1d\c8\b64e19423cc5a2a339450ea5d145e7c8eb3d4aa2b150cde33b
Successfully built wikipedia

   ------------------------ --------------- 3/5 [anthropic]
   ------------------------ --------------- 3/5 [anthropic]
   ------------------------ --------------- 3/5 [anthropic]
   ------------------------ --------------- 3/5 [anthropic]
   ---------------------------------------- 5/5 [langchain-anthropic]

Note: you may need to restart the kernel to use updated packages.


  DEPRECATION: Building 'wikipedia' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'wikipedia'. Discussion can be found at https://github.com/pypa/pip/issues/6334
